In [ ]:
import pyarrow  # Load this first
import pandas as pd
import numpy as np
import glob
import pickle
import py7zr
import os
from tqdm import tqdm
from scipy.spatial import cKDTree
from infostop import Infostop

def location_infostop_optimized(df, r1=100, r2=100, min_staying_time=300, max_time_between=43200,
                                min_spatial_resolution=1e-5, weighted=True, weight_exponent=1):
    """
    Optimized to avoid groupby.apply() and use vectorized splitting.
    """
    # 1. Preprocess using vectorized filtering
    # criteria: (dwell_time is NaN OR > 60000) AND (speed is NaN OR < 10)
    mask = (df['dwell_time'].isna() | (df['dwell_time'] > 60000)) & \
           (df['speed'].isna() | (df['speed'] < 10))
    df = df[mask].copy()

    # 2. Vectorized Sorting and Trajectory Extraction
    # Instead of groupby.apply, sort once and find boundaries
    df.sort_values(['advertiser_id', 'location_at'], inplace=True)
    
    # Find indices where the advertiser_id changes
    ids = df['advertiser_id'].values
    change_idx = np.where(ids[:-1] != ids[1:])[0] + 1
    
    # Split into list of numpy arrays for Infostop
    X = np.split(df[['latitude', 'longitude', 'location_at']].values, change_idx)

    # 3. Infostop Model
    model = Infostop(r1=r1, r2=r2, min_staying_time=min_staying_time, 
                     max_time_between=max_time_between,
                     min_spacial_resolution=min_spatial_resolution, 
                     weighted=weighted, weight_exponent=weight_exponent)

    stop_ids = np.concatenate(model.fit_predict(X))
    df['stop_id'] = stop_ids
    
    # 4. Label Medians (Vectorized mapping)
    label_medians = model.compute_label_medians()
    # Map coordinates back to a median frame efficiently
    median_df = pd.DataFrame.from_dict(label_medians, orient='index', columns=['lat', 'lon'])
    median_df.index.name = 'label'
    
    return df, median_df


def process_7z_archive(archive_path):
    """
    Extracts and processes files one by one to save memory.
    """
    with py7zr.SevenZipFile(archive_path, mode='r') as archive:
        # Get list of CSV files in the 7z
        targets = [f for f in archive.getnames() if f.endswith('.csv')]
        
        for filename in targets:
            print(f"Processing {filename}...")
            # Extract single file to a temp folder
            archive.extract(targets=[filename], path='temp_extract')
            file_path = os.path.join('temp_extract', filename)
            
            # Load and optimize
            df = pd.read_csv(file_path, low_memory=False)
            df_processed, medians = location_infostop_optimized(df)
            
            # Save results (Using Parquet is much faster than CSV.XZ)
            out_name = file_path.replace(".csv", "_stop.parquet")
            df_processed.to_parquet(out_name, index=False, engine='fastparquet')
            
            # Clean up temp file
            os.remove(file_path)


archive_file = '2019_09.7z'
if os.path.exists(archive_file):
    process_7z_archive(archive_file)


Processing 09/Pittsburg_Sep01_2019.csv...


In [17]:
import pyarrow  # Load this first
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import glob

def match_gps_overpoi_nearest(pois, users, dist_threshold=0.001):
    """
    Finds the nearest POI for each user ping within a threshold.
    Ensures one ping only matches the closest POI.
    """
    user_coords = users[['latitude', 'longitude']].values
    poi_coords = pois[['latitude', 'longitude']].values
    tree = cKDTree(user_coords)
    
    # Query for matches within threshold
    indices = tree.query_ball_point(poi_coords, r=dist_threshold)
    
    results = []
    for i, user_idx_list in enumerate(indices):
        if not user_idx_list:
            continue
            
        current_poi = pois.iloc[i]
        poi_loc = poi_coords[i]
        
        # Get all potential user matches for this POI
        potential_matches = users.iloc[user_idx_list].copy()
        
        # Calculate exact distance
        u_locs = potential_matches[['latitude', 'longitude']].values
        dists = np.linalg.norm(u_locs - poi_loc, axis=1)
        
        potential_matches['dist_to_poi'] = dists
        potential_matches['business_id'] = current_poi['business_id']
        
        results.append(potential_matches)

    if not results:
        return pd.DataFrame()

    all_candidates = pd.concat(results)
    
    # Deduplicate: Keep the closest POI for any specific ping
    best_matches = all_candidates.sort_values('dist_to_poi').drop_duplicates(
        subset=['advertiser_id', 'location_at'], 
        keep='first'
    )
    
    return best_matches

def get_visit_sequences(poi_path, stop_files_pattern):
    # 1. Load POIs (Minimal columns for matching)
    poi = pd.read_csv(poi_path)[['business_id', 'latitude', 'longitude']]
    
    all_sequences = []
    stop_files = glob.glob(stop_files_pattern)

    for file in stop_files:
        df = pd.read_parquet(file,engine="fastparquet")
        
        # Filter for identified stops
        stops_only = df[df['stop_id'] != -1].copy()
        if stops_only.empty: 
            continue

        # 2. Use the improved matching function
        visit_pings = match_gps_overpoi_nearest(poi, stops_only, dist_threshold=0.001)
        
        if visit_pings.empty: 
            continue

        # 3. Collapse pings into Visit Events
        visits = visit_pings.groupby(['advertiser_id', 'stop_id', 'business_id']).agg(
            arrival_time=('location_at', 'min'),
            departure_time=('location_at', 'max'),
            ping_count=('location_at', 'count'),
            avg_dist_to_poi=('dist_to_poi', 'mean')
        ).reset_index()

        all_sequences.append(visits)

    if not all_sequences:
        return pd.DataFrame()

    full_visit_df = pd.concat(all_sequences).sort_values(['advertiser_id', 'arrival_time'])
    return full_visit_df

# Generate the master visit list
visits_master = get_visit_sequences("./data/poi_pittsburg.csv", './temp_extract/09/*.parquet')

In [14]:
glob.glob('./temp_extract/*.parquet')

[]

In [18]:
visits_master

,advertiser_id,stop_id,business_id,arrival_time,departure_time,ping_count,avg_dist_to_poi
0,00002443-710B-4713-9AA9-511B0EA8C560,10956,9UzrPAKKnwao1GbcaBoxtQ,1567352063,1567352666,4,0.000096
0,00002443-710B-4713-9AA9-511B0EA8C560,12,9IhBQ7mTYajCQtIrEjv5KQ,1567535396,1567536100,2,0.000219
1,00002443-710B-4713-9AA9-511B0EA8C560,12,H6skMpg_g-sOrfTQf_Q5LQ,1567537250,1567537250,1,0.000273
0,00002443-710B-4713-9AA9-511B0EA8C560,211,J7r1lw3Yv9PnXC5EHH4CUQ,1567605105,1567605105,1,0.000525
3,00002443-710B-4713-9AA9-511B0EA8C560,1923,VFnDUlklzBkal6Tlv0ns0w,1567614630,1567614630,1,0.000271
...,...,...,...,...,...,...,...
104887,FFFBF545-3958-4612-9049-AF679A803BF8,484,gu_yLQWygwzkwBOCpSj-Kg,1567626430,1567626430,1,0.000187
104886,FFFBF545-3958-4612-9049-AF679A803BF8,484,UyEJouFTFxcmYaHouhSGFA,1567626475,1567626995,12,0.000259
104885,FFFBF545-3958-4612-9049-AF679A803BF8,424,U9NeWizXL6V49KElTXW6mA,1567628373,1567629695,4,0.000501
91658,FFFD4AB5-4FF7-4741-BDEB-B7CD2F88CABE,2,4RNBXjX2kTwpawbmDQqzow,1567296232,1567303708,16,0.000791


In [21]:
import pyarrow  # Load this first
import pandas as pd
df = pd.read_parquet('visits_master_2019_12.parquet', engine = 'fastparquet')

In [22]:
df

,advertiser_id,stop_id,business_id,arrival_time,departure_time,ping_count,avg_dist_to_poi
0,00002443-710B-4713-9AA9-511B0EA8C560,436,Tscoq9kue8_J1ySO42_ItA,1575300425,1575300932,2,0.000304
1,00002443-710B-4713-9AA9-511B0EA8C560,4,GN0LE1tKNoT0UeWrUU-cbA,1575367423,1575367424,2,0.000485
2,00002443-710B-4713-9AA9-511B0EA8C560,4,yFumR3CWzpfvTH2FCthvVw,1575367730,1575367730,1,0.000376
3,00002443-710B-4713-9AA9-511B0EA8C560,1748,-y8ISN5r7YiRZp7n7OHKTA,1575549833,1575549833,1,0.000348
4,00002443-710B-4713-9AA9-511B0EA8C560,1748,thCqq5IrLTZzb5FbYm-6DQ,1575550134,1575550134,1,0.000141
...,...,...,...,...,...,...,...
1156990,FFFE35D8-23F1-4173-873B-D6CC7795F861,495,dKxeNoetk9TMipSnBbamCw,1576355311,1576355311,1,0.000393
1156991,FFFFE0E2-669C-43A4-B098-774F6A5C686A,500,jqBUVAWJRJ6QIGXVuhdVvw,1575486103,1575486103,1,0.000456
1156992,FFFFE0E2-669C-43A4-B098-774F6A5C686A,500,asRR7Nw3CI4lJB9696crmw,1575486149,1575490535,8,0.000307
1156993,FFFFE0E2-669C-43A4-B098-774F6A5C686A,500,EafZ9d170S18zqd1qs9MLQ,1575487578,1575487578,1,0.000430


In [24]:
import pandas as pd

# 1. Preprocessing
# Convert timestamps to datetime
df['arrival_time'] = pd.to_datetime(df['arrival_time'], unit='s')
df['arrival_time'] = df['arrival_time'].dt.tz_localize('UTC').dt.tz_convert('US/Eastern')

# Create derived columns
df['month'] = df['arrival_time'].dt.to_period('M')
df['hour'] = df['arrival_time'].dt.hour
df['date'] = df['arrival_time'].dt.date
df['weekday'] = df['arrival_time'].dt.weekday < 5  # True for Mon-Fri

# Define Lunch (e.g., 11 AM - 2 PM) and Dinner (e.g., 5 PM - 9 PM)
df['is_lunch'] = (df['hour'] >= 11) & (df['hour'] < 14)
df['is_dinner'] = (df['hour'] >= 17) & (df['hour'] < 21)

# Helper function to format the output row
def get_stats(series):
    desc = series.describe()
    return {
        'Mean': desc['mean'],
        'SD': desc['std'],
        'P25': desc['25%'],
        'Median': desc['50%'],
        'P75': desc['75%'],
        'Min': desc['min'],
        'Max': desc['max']
    }

stats_rows = []

# --- PANEL A: Consumer Behavior ---

# 1. Total visits per consumer (overall)
s = df.groupby('advertiser_id').size()
stats_rows.append({'Variable': 'Total visits per consumer (overall)', **get_stats(s)})

# 2. Unique POIs visited (overall)
s = df.groupby('advertiser_id')['business_id'].nunique()
stats_rows.append({'Variable': 'Unique POIs visited (overall)', **get_stats(s)})

# 3. Monthly visits per consumer
s = df.groupby(['advertiser_id', 'month']).size()
stats_rows.append({'Variable': 'Monthly visits per consumer', **get_stats(s)})

# 4. Monthly unique POIs per consumer
s = df.groupby(['advertiser_id', 'month'])['business_id'].nunique()
stats_rows.append({'Variable': 'Monthly unique POIs per consumer', **get_stats(s)})

# Note: "Home CBG population" cannot be calculated from the provided snippet.

# --- PANEL B: Trajectory Patterns ---

# 5. Monthly visits per restaurant
s = df.groupby(['business_id', 'month']).size()
stats_rows.append({'Variable': 'Monthly visits per restaurant', **get_stats(s)})

# 6. Monthly lunch visits to a restaurant
lunch_df = df[df['is_lunch']]
if not lunch_df.empty:
    s = lunch_df.groupby(['business_id', 'month']).size()
    # Reindex with all restaurant-months to include 0s if strictly required, 
    # but usually "visits to a restaurant" implies active counts. 
    stats_rows.append({'Variable': 'Monthly lunch visits to a restaurant', **get_stats(s)})

# 7. Monthly dinner visits to a restaurant
dinner_df = df[df['is_dinner']]
if not dinner_df.empty:
    s = dinner_df.groupby(['business_id', 'month']).size()
    stats_rows.append({'Variable': 'Monthly dinner visits to a restaurant', **get_stats(s)})

# 8. Visits per active day (User intensity)
# (Total Visits / Number of Days Visited) per User
user_visits = df.groupby('advertiser_id').size()
user_days = df.groupby('advertiser_id')['date'].nunique()
s = user_visits / user_days
stats_rows.append({'Variable': 'Visits per active day', **get_stats(s)})

# 9. Share of weekday visits (%) - Per Restaurant
# Calculate percentage of weekday visits for each restaurant
rest_total = df.groupby('business_id').size()
rest_weekday = df[df['weekday']].groupby('business_id').size()
# Ensure restaurants with 0 weekday visits are counted as 0
rest_weekday = rest_weekday.reindex(rest_total.index, fill_value=0)
s = (rest_weekday / rest_total) * 100
stats_rows.append({'Variable': 'Share of weekday visits (%)', **get_stats(s)})

# 10. Share of lunch visits (%) - Per Restaurant
rest_lunch = df[df['is_lunch']].groupby('business_id').size()
rest_lunch = rest_lunch.reindex(rest_total.index, fill_value=0)
s = (rest_lunch / rest_total) * 100
stats_rows.append({'Variable': 'Share of lunch visits (%)', **get_stats(s)})

# 11. Travel distance for lunch (miles)
# Distribution of distances for all lunch visits
if not lunch_df.empty:
    s = lunch_df['avg_dist_to_poi']
    stats_rows.append({'Variable': 'Travel distance for lunch (miles)', **get_stats(s)})

# 12. Travel distance for dinner (miles)
if not dinner_df.empty:
    s = dinner_df['avg_dist_to_poi']
    stats_rows.append({'Variable': 'Travel distance for dinner (miles)', **get_stats(s)})

# Compile Final Table
result_table = pd.DataFrame(stats_rows)
print(result_table[['Variable', 'Mean', 'SD', 'P25', 'Median', 'P75', 'Min', 'Max']].round(1))

/tmp/ipykernel_1860460/1709520987.py:9: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['month'] = df['arrival_time'].dt.to_period('M')


                                 Variable  Mean     SD   P25  Median   P75  \
0     Total visits per consumer (overall)  14.8   20.3   3.0     7.0  18.0   
1           Unique POIs visited (overall)  10.2   11.4   3.0     6.0  13.0   
2             Monthly visits per consumer  13.6   19.5   2.0     6.0  16.0   
3        Monthly unique POIs per consumer   9.4   11.0   2.0     5.0  12.0   
4           Monthly visits per restaurant  96.1  202.0   4.0    28.0  97.0   
5    Monthly lunch visits to a restaurant  33.6   57.4   6.0    14.0  36.0   
6   Monthly dinner visits to a restaurant  29.5   64.8   2.0     9.0  29.0   
7                   Visits per active day   3.5    2.6   1.9     2.9   4.3   
8             Share of weekday visits (%)  72.8   14.0  65.0    73.3  81.4   
9               Share of lunch visits (%)  20.0   11.0  13.2    19.5  25.9   
10      Travel distance for lunch (miles)   0.0    0.0   0.0     0.0   0.0   
11     Travel distance for dinner (miles)   0.0    0.0   0.0    

In [25]:
import pandas as pd
df = pd.read_csv("09/Pittsburg_Sep02_2019.csv")

In [26]:
df

,advertiser_id,platform,latitude,longitude,horizontal_accuracy,location_at,email,ipv_4,user_agent,final_country,...,device_model,wifi_ssid,wifi_bssid,decorated_at,dist_moved,day_number,day_type,time_type,dwell_type,tech_signals
0,E8F113DE-B4CB-4F80-809B-8038ED842DE1,IDFA,40.508513,-80.248968,10.5,1567430638,NaN,NaN,12.4,US,...,"iPhone11,8",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0BA7839F-8604-4E4D-9C87-314DC13B17DB,IDFA,40.311155,-79.617877,10.0,1567449288,NaN,NaN,12.4,US,...,"iPhone10,3",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,18655EAF-06E4-4566-9660-C51850D6FAE8,IDFA,40.360856,-80.079100,10.0,1567450910,NaN,NaN,12.3.2,US,...,"iPhone10,2",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,EA2ED5CF-BCB6-4D91-8A87-1BF716C568AE,IDFA,40.370185,-80.224228,65.0,1567422260,NaN,NaN,13.1,US,...,"iPhone11,6",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,11601D88-E2B1-49D5-B40D-8FD59417A223,AAID,40.689206,-80.290936,15.8,1567439314,NaN,NaN,8.0.0,US,...,SAMSUNG-SM-G930A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10911070,77F46B2B-C1BE-4959-BCF4-33E5DA054BFC,AAID,40.630506,-79.881004,4.3,1567442203,NaN,NaN,9,US,...,SM-G960U,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10911071,4C5D21B5-1EF6-4B37-AC33-3EC59909788D,IDFA,40.742303,-80.293513,65.0,1567460757,NaN,NaN,12.4,US,...,"iPhone10,2",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10911072,FAD3724B-3BB5-4A09-871B-07AA154358F1,IDFA,40.710127,-80.103854,10.0,1567396725,NaN,NaN,12.3.1,US,...,"iPhone10,1",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10911073,121627A5-1636-4647-979D-CABFF24971C4,IDFA,40.326255,-79.695670,5.0,1567456202,NaN,NaN,12.4,US,...,"iPhone10,1",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
import pandas as pd

# --- NEW: Filtering for Restaurants ---
# This ensures we only analyze rows where 'Restaurants' is in the category string
df = pd.read_csv("data/poi_pittsburg.csv")
df = df[df['categories'].str.contains('Restaurants', na=False, case=False)].copy()

# # 1. Preprocessing
# df['arrival_time'] = pd.to_datetime(df['arrival_time'], unit='s')
# df['arrival_time'] = df['arrival_time'].dt.tz_localize('UTC').dt.tz_convert('US/Eastern')

# # Create derived columns
# df['month'] = df['arrival_time'].dt.to_period('M')
# df['hour'] = df['arrival_time'].dt.hour
# df['date'] = df['arrival_time'].dt.date
# df['weekday'] = df['arrival_time'].dt.weekday < 5 

# df['is_lunch'] = (df['hour'] >= 11) & (df['hour'] < 14)
# df['is_dinner'] = (df['hour'] >= 17) & (df['hour'] < 21)

# # ... [Rest of your helper functions and Panel A/B logic remains the same] ...

In [33]:
df

,Unnamed: 0,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
5,226,sm5Sl7z9Kx1yXExrdGuZ7g,Cho Buffet,6302 Robinson Centre Dr,Pittsburgh,PA,15205.0,40.452644,-80.160241,1.5,7,0,{'RestaurantsPriceRange2': '2'},"Buffets, Restaurants","{'Monday': '11:0-22:30', 'Tuesday': '11:0-22:3..."
6,244,goar7zF4G0LdsQ1Y4KS3Iw,Pittsburgh Barbecue Company,1000 Banksville Rd,Pittsburgh,PA,15216.0,40.419899,-80.030052,4.0,91,1,"{'WiFi': ""'no'"", 'RestaurantsPriceRange2': '2'...","Caterers, Event Planning & Services, Restauran...","{'Wednesday': '11:0-20:0', 'Thursday': '11:0-2..."
7,302,grZ6FnfZoj1pQWElAQve3g,Wendy's,2410 W. Liberty,Pittsburgh,PA,15226.0,40.400235,-80.024127,2.0,22,1,"{'RestaurantsPriceRange2': '1', 'BikeParking':...","Fast Food, Restaurants, Burgers","{'Monday': '10:0-2:0', 'Tuesday': '10:0-2:0', ..."
9,383,1YC7AbQMlNb5mLUlwwaa3w,Bites and Brews,5750 Ellsworth Ave,Pittsburgh,PA,15232.0,40.455696,-79.930883,3.5,85,0,"{'GoodForKids': 'False', 'BusinessAcceptsCredi...","Pizza, Barbeque, Chicken Wings, Nightlife, Res...","{'Monday': '16:0-23:0', 'Tuesday': '16:0-23:0'..."
17,544,y-0Twdg0sIhrh7BFJii8LA,Duquesne Club,325 6th Ave,Pittsburgh,PA,15222.0,40.441961,-79.998665,5.0,3,1,{'GoodForKids': 'False'},"Arts & Entertainment, Social Clubs, Signature ...","{'Monday': '7:0-22:0', 'Tuesday': '7:0-22:0', ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7619,209061,8Jub_J2ILU0xYYnsgcd7UQ,Silvi's Southside Kitchen,2212 E Carson St,Pittsburgh,PA,15203.0,40.428108,-79.973988,4.0,4,0,"{'RestaurantsTakeOut': 'True', 'Ambience': ""{'...","Mexican, Food, Restaurants, American (New)","{'Monday': '23:0-21:0', 'Tuesday': '23:0-21:0'..."
7620,209101,gB7X5o33o6W_rAGgHunSCA,McDonald's,4849 Mcknight Rd,Pittsburgh,PA,15237.0,40.528601,-80.008328,2.0,10,1,"{'RestaurantsPriceRange2': '1', 'RestaurantsGo...","Fast Food, Restaurants, Coffee & Tea, Food, Bu...","{'Monday': '4:0-23:0', 'Tuesday': '4:0-23:0', ..."
7624,209184,gHngt6zpP683GKe1i23LUg,Grand Concourse,100 W Station Square Dr,Pittsburgh,PA,15219.0,40.433716,-80.003919,3.5,451,1,"{'RestaurantsReservations': 'True', 'Restauran...","Restaurants, Seafood, Breakfast & Brunch, Amer...","{'Monday': '11:0-22:0', 'Tuesday': '11:0-22:0'..."
7625,209223,Y-vxPbvPKcaLiYa9ZBXoDg,McDonald's,505 Smithfield St,Pittsburgh,PA,15219.0,40.440433,-79.998637,1.5,24,0,"{'OutdoorSeating': 'False', 'RestaurantsTakeOu...","Restaurants, Burgers, Hot Dogs, Food, Coffee &...","{'Monday': '5:0-20:0', 'Tuesday': '5:0-20:0', ..."
